In [ ]:
# ==========================================
# 1. 自動安裝所需的第三方套件
# ==========================================
print("正在安裝必要套件...")
!pip install Flask pyngrok line-bot-sdk requests google-genai --quiet
print("✅ 套件安裝完成！")

# ==========================================
# 2. 匯入模組與讀取 Colab 秘密鑰匙 (Secrets)
# ==========================================
import os
import requests
from flask import Flask, request, abort
from pyngrok import ngrok
from google.colab import userdata

# 讀取環境變數（請確保 Colab 左側「鑰匙」圖示內有設定這些變數）
ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
google_gemini_api_key = userdata.get('GEMINI_API_KEY') # 此範例暫未用到
port = 5051

# ==========================================
# 3. 清理舊連線並啟動 Ngrok 隧道
# ==========================================
print("正在清理舊的 Ngrok 連線並重新建立隧道...")
ngrok.kill() # 強制關閉舊隧道，避免超過免費版連線數限制

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(port, name="linebot_tunnel")
webhook_url = tunnel.public_url
print(f"🚀 Ngrok 隧道建立成功！公網網址為: {webhook_url}")

# ==========================================
# 4. 自動將新網址更新至 LINE 官方後台
# ==========================================
def update_line_webhook(url_to_update):
    """使用 LINE Messaging API 自動更新 Webhook Endpoint"""
    api_url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {"endpoint": url_to_update}

    response = requests.put(api_url, headers=headers, json=data)
    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已成功自動同步為：{url_to_update}")
        return True
    else:
        print(f"❌ LINE Webhook 更新失敗：{response.status_code} - {response.text}")
        return False

update_line_webhook(webhook_url)

# ==========================================
# 5. 初始化 LINE Bot v3 SDK 與 Flask 伺服器
# ==========================================
from linebot.v3 import WebhookHandler
from linebot.v3.exceptions import InvalidSignatureError
from linebot.v3.messaging import Configuration, ApiClient, MessagingApi, ReplyMessageRequest, TextMessage
from linebot.v3.webhooks import MessageEvent, TextMessageContent

app = Flask(__name__)
configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)

@app.route("/", methods=['POST'])
def callback():
    # 取得 LINE 傳來的數位簽章，用以驗證請求是否合法
    signature = request.headers['X-Line-Signature']
    body = request.get_data(as_text=True)
    print("收到 LINE Webhook Body: ", body)

    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("數位簽章驗證失敗，請檢查 Token 與 Secret 設定。")
        abort(400)
    return 'OK'

# 當收到「文字訊息」時的處理邏輯
@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    print("收到訊息事件 Event: ", event)

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        # 0501作業重點：為什麼 LINE 會回應兩次？
        # 在 ReplyMessageRequest 的 messages 陣列參數中，我們同時傳入了兩個 TextMessage 物件。
        # LINE Messaging API 允許在單次的回覆（同一個 reply_token）中打包發送最多 5 則訊息。
        # 因為這裡陣列內重複寫了兩次 `TextMessage(text=event.message.text)`，
        # 所以當使用者傳送一條訊息過來時，機器人就會在同一個時間點連續吐回兩條一模一樣的訊息。
        line_bot_api.reply_message_with_http_info(
            ReplyMessageRequest(
                reply_token=event.reply_token,
                messages=[
                    TextMessage(text=event.message.text), # 第一條回覆訊息
                    TextMessage(text=event.message.text)  # 第二條回覆訊息（造成回應兩次的主因）
                ]
            )
        )

# ==========================================
# 6. 啟動 Flask 服務
# ==========================================
if __name__ == "__main__":
    print(f"正在本機 Port {port} 啟動 Flask 伺服器...")
    app.run(port=port)

正在安裝必要套件...
✅ 套件安裝完成！
正在清理舊的 Ngrok 連線並重新建立隧道...
🚀 Ngrok 隧道建立成功！公網網址為: https://strode-clothes-crested.ngrok-free.dev
✅ LINE Webhook URL 已成功自動同步為：https://strode-clothes-crested.ngrok-free.dev
正在本機 Port 5051 啟動 Flask 伺服器...
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
